# Food-91 Classification with Custom CNN

A deep learning project that classifies food images into **91 categories** using a custom-built CNN with **Squeeze-and-Excitation attention** and **residual connections**, then generates a user taste profile using an LLM.

## Highlights
- Custom CNN architecture (not pretrained) — trained from scratch
- SE-Net attention mechanism for channel-wise feature recalibration
- Residual blocks for stable deep training
- Weighted sampling to handle class imbalance
- **51.32% test accuracy** on 91 food categories
- LLM-powered taste profile generation from predicted categories

## Requirements
```
torch >= 2.6
torchvision
numpy
google-genai  # for bonus LLM feature
```

## 1. Setup & Data Loading

Download the [Food-91 dataset](https://github.com/MaritPaul/Neural-Computing-datasets) and organize it as:
```
datasets/
├── train/
│   ├── apple_pie/
│   ├── baby_back_ribs/
│   └── ...  (91 classes)
└── test/
    ├── apple_pie/
    ├── baby_back_ribs/
    └── ...  (91 classes)
```

Update `train_path` and `test_path` below to point to your local dataset.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import numpy as np
import random
from collections import Counter
import torch.nn.functional as F

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Reproducibility ──
def set_seed(seed=64):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(64)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ── Dataset paths (update these) ──
train_path = "./datasets/train"  # <-- your train folder
test_path  = "./datasets/test"   # <-- your test folder

# ── Transforms ──
image_size = 96

train_transform = transforms.Compose([
    transforms.Resize((image_size + 16, image_size + 16)),
    transforms.RandomResizedCrop(image_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

# ── Load datasets ──
train_ds = datasets.ImageFolder(root=train_path, transform=train_transform)
test_ds  = datasets.ImageFolder(root=test_path,  transform=test_transform)

# ── Weighted sampler for class imbalance ──
labels = train_ds.targets
w = 1.0 / np.bincount(labels)
sample_w = torch.DoubleTensor(w[labels])
sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

# ── DataLoaders ──
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,  num_workers=0)

print(f"Classes: {len(train_ds.classes)}")
print(f"Train: {len(train_ds)} images | Test: {len(test_ds)} images")
print("Dataset loaded successfully.")

## 2. CNN Architecture

Custom CNN with:
- **Squeeze-and-Excitation (SE) blocks** — channel attention that learns which feature maps matter most
- **Residual connections** — enables stable training of deeper networks
- **Architecture**: Stem (Conv→BN→ReLU→Pool) → 3 Residual Stages (128→256→512) → Global Average Pool → FC Head

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block for channel attention."""
    def __init__(self, c, r=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),          # [B,C,H,W] -> [B,C,1,1]
            nn.Conv2d(c, c // r, 1), nn.ReLU(),
            nn.Conv2d(c // r, c, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x)


class ResidualBlock(nn.Module):
    """Residual block with optional downsampling and SE attention."""
    def __init__(self, in_ch, out_ch, downsample=False):
        super().__init__()
        s = 2 if downsample else 1
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, s, 1), nn.BatchNorm2d(out_ch), nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1), nn.BatchNorm2d(out_ch),
            SEBlock(out_ch)
        )
        self.shortcut = (
            nn.Identity() if (in_ch == out_ch and not downsample)
            else nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, s), nn.BatchNorm2d(out_ch))
        )

    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))


class FoodCNN(nn.Module):
    def __init__(self, num_classes=91):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.layer1 = ResidualBlock(64,  128, downsample=True)
        self.layer2 = ResidualBlock(128, 256, downsample=True)
        self.layer3 = ResidualBlock(256, 512, downsample=True)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.fc(self.pool(x))


# ── Initialize ──
model = FoodCNN(num_classes=len(train_ds.classes)).to(device)

optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4
)

def accuracy(output, target):
    pred = output.argmax(1)
    return (pred == target).sum().item(), target.size(0)

print(model)

## 3. Training

Training loop with early stopping (patience=15) and learning rate scheduling.

In [ ]:
set_seed(64)
torch.use_deterministic_algorithms(True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

best_acc = 0.0
patience = 15
no_gain = 0

for epoch in range(100):
    # ── Train ──
    model.train()
    train_correct, train_total = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        c, n = accuracy(out, yb)
        train_correct += c
        train_total += n

    # ── Evaluate ──
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c, n = accuracy(model(xb), yb)
            test_correct += c
            test_total += n

    train_acc = 100 * train_correct / train_total
    test_acc  = 100 * test_correct / test_total
    scheduler.step(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        no_gain = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ★ Epoch {epoch+1:02d}  new best: {best_acc:.2f}%")
    else:
        no_gain += 1

    if no_gain >= patience:
        print(f"  Early stopped at epoch {epoch+1}")
        break

    print(f"[{epoch+1:02d}] Train {train_acc:.2f}% | Test {test_acc:.2f}% | LR {optimizer.param_groups[0]['lr']:.1e}")

print(f"\nBest Test Accuracy: {best_acc:.2f}%")

 Epoch 01 new best 9.87%
[01] Train 6.17% | Test 9.87% | LR 5.0e-04


KeyboardInterrupt: 

## 4. Evaluation

Load the best saved model and report final test accuracy.

In [ ]:
def calculate_test_accuracy(model):
    model.eval()
    results = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            results.append(accuracy(model(data), target))
    correct, total = map(sum, zip(*results))
    return (correct / total) * 100


model = FoodCNN().to(device)
model.load_state_dict(torch.load("best_model.pth"))

final_acc = calculate_test_accuracy(model)
print(f"Final Test Accuracy: {final_acc:.2f}%")

Final Test Accuracy: 51.32%


## 5. Hyperparameters

| Parameter | Value |
|-----------|-------|
| Learning rate | 5e-4 |
| Batch size | 32 |
| Max epochs | 100 (early stop patience 15) |
| Optimizer | AdamW (weight decay 3e-4) |
| Loss | CrossEntropyLoss (label smoothing 0.2) |
| LR scheduler | ReduceLROnPlateau (factor 0.5, patience 4) |
| Image size | 96 × 96 |
| Augmentation | RandomResizedCrop, HorizontalFlip |
| Dropout | 0.5 |
| Network depth | ~23 layers (conv + BN + SE + FC) |

## 6. User Simulation

Pick 10 random test images to simulate a user uploading food photos, then report predicted categories.

In [ ]:
def simulate_user(show_results=False):
    random.seed(42)
    indices = random.sample(range(len(test_ds)), 10)

    model.eval()
    predicted = []
    with torch.no_grad():
        for idx in indices:
            image, _ = test_ds[idx]
            image = image.unsqueeze(0).to(device)
            output = model(image)
            cls = torch.argmax(output, dim=1).item()
            predicted.append(cls)

    names = [train_ds.classes[i] for i in predicted]
    counts = Counter(names)

    if show_results:
        print("Predicted Taste Profile (10 images):")
        for cls, freq in counts.items():
            print(f"  {cls}: {freq}")
    return names

simulate_user(show_results=True)

Predicted Taste Profile (10 images):
- grilled_salmon: frequency 1
- onion_rings: frequency 1
- poutine: frequency 1
- french_onion_soup: frequency 1
- hamburger: frequency 1
- escargots: frequency 1
- clam_chowder: frequency 1
- chicken_curry: frequency 1
- grilled_cheese_sandwich: frequency 1
- filet_mignon: frequency 1


## 7. Bonus: LLM Taste Profile Generation

Use the Gemini API to generate a natural-language description of the user's food preferences based on the 10 predicted categories.

Set your API key as an environment variable:
```bash
export GEMINI_API_KEY="your-key-here"
```

In [ ]:
from google import genai

# Load API key from environment variable
api_key = os.environ.get("GEMINI_API_KEY", "")
assert api_key, "Set GEMINI_API_KEY environment variable first."

client = genai.Client(api_key=api_key)

# Get random test images
random_indices = random.sample(range(len(test_ds)), 10)
image_paths = [test_ds.samples[i][0] for i in random_indices]

# Identify each food image with Gemini
keywords = []
for path in image_paths:
    uploaded = client.files.upload(file=path)
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[uploaded, "Name it with one word what you see."],
    )
    keywords.append(response.text.strip())

# Generate taste profile
profile = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=f"Generate a short description of the food preference based on these 10 keywords: {' '.join(keywords)}",
)

print(profile.text)

This person enjoys a diverse palate, spanning comfort food classics like **soup, chowder, fish and chips,** and **apple pie**, to more globally-inspired flavors with **curry** and **escargot**. They appreciate seafood, indicated by **shrimp** and a likely broader enjoyment of fish, and enjoy both savory meat dishes featuring **wings** and **pork**, as well as sweet treats like **cake.**



### Example Output

> *This person enjoys a diverse range of foods, spanning from seafood like shrimp, fish and chips, and chowder to savory dishes like wings, curry, and pork chops. They also have a liking for classic comfort food such as pie and soup, but with an adventurous side as shown by their preference for escargots. Finally, they enjoy some sweets like cake.*